## S3 Landing → Bronze Delta
This notebook ingests raw files from Amazon S3, applies light conditioning in the bronze layer, and persists a managed Delta table in Unity Catalog.

### Notebook widgets
Provide the S3 source information plus the Unity Catalog destination for the bronze Delta table.

In [0]:
dbutils.widgets.text("s3_source", "s3://my-demo-bucket/raw/events", "S3 Source Prefix")
dbutils.widgets.text("aws_region", "", "AWS Region (optional)")
dbutils.widgets.dropdown("file_format", "json", ["json", "csv", "parquet"], "Source Format")
dbutils.widgets.text("schema_location", "dbfs:/schemas/bronze/s3_events", "Auto Loader Schema Path")
dbutils.widgets.text("checkpoint_path", "dbfs:/checkpoints/bronze/s3_events", "Checkpoint Path")
dbutils.widgets.text("catalog", "main", "Catalog")
dbutils.widgets.text("schema", "skew_demo", "Schema")
dbutils.widgets.text("table_name", "events_bronze", "Bronze Table")
dbutils.widgets.dropdown("trigger_once", "true", ["true", "false"], "Trigger Once")
dbutils.widgets.text("max_files_per_trigger", "1000", "Max Files / Trigger")

### Imports & Spark configuration

In [0]:
from typing import Iterable

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import (
    DoubleType,
    StringType,
    StructField,
    StructType,
)

spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.sql.files.ignoreCorruptFiles", "true")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "200")

### Helpers for schema + bronze conditioning

In [0]:

def bronze_schema() -> StructType:
    """Default schema for semi-structured transaction-like events."""
    return StructType(
        [
            StructField("event_ts", StringType(), True),
            StructField("transaction_id", StringType(), True),
            StructField("customer_id", StringType(), True),
            StructField("merchant_id", StringType(), True),
            StructField("region", StringType(), True),
            StructField("amount", StringType(), True),
            StructField("currency", StringType(), True),
            StructField("status", StringType(), True),
            StructField("channel", StringType(), True),
        ]
    )


def to_sha2(columns: Iterable[str]):
    """Use a deterministic SHA-256 hash over all columns to detect duplicates."""
    return F.sha2(
        F.concat_ws("||", *[F.col(col).cast("string") for col in columns]),
        256,
    )


def apply_bronze_transformations(df: DataFrame) -> DataFrame:
    """Bronze-level conditioning: type casting, metadata, and deduplication."""
    conditioned = (
        df.withColumn("event_ts", F.to_timestamp("event_ts"))
        .withColumn("amount", F.col("amount").cast(DoubleType()))
        .withColumn("region", F.upper(F.trim(F.col("region"))))
        .withColumn("_ingest_ts", F.current_timestamp())
        .withColumn("_ingest_date", F.to_date(F.col("_ingest_ts")))
        .withColumn("_source_file", F.input_file_name())
        .withColumn("_raw_checksum", to_sha2(df.columns))
    )

    filtered = conditioned.filter(
        F.col("event_ts").isNotNull() & F.col("transaction_id").isNotNull()
    )

    return filtered.dropDuplicates(["transaction_id", "event_ts"])


### Build Auto Loader stream from S3

In [0]:
source_path = dbutils.widgets.get("s3_source").rstrip("/")
aws_region = dbutils.widgets.get("aws_region").strip()
file_format = dbutils.widgets.get("file_format")
schema_location = dbutils.widgets.get("schema_location")
checkpoint_path = dbutils.widgets.get("checkpoint_path")
catalog = dbutils.widgets.get("catalog")
schema_name = dbutils.widgets.get("schema")
bronze_table = f"`{catalog}`.`{schema_name}`.`{dbutils.widgets.get('table_name')}`"
trigger_once = dbutils.widgets.get("trigger_once") == "true"
max_files_per_trigger = dbutils.widgets.get("max_files_per_trigger")

reader = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", file_format)
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("cloudFiles.maxFilesPerTrigger", max_files_per_trigger)
)

if aws_region:
    reader = reader.option("cloudFiles.region", aws_region)

if file_format == "csv":
    reader = reader.option("header", "true").option("escape", '"')

reader = reader.schema(bronze_schema()).load(source_path)

bronze_df = apply_bronze_transformations(reader)

write_stream = (
    bronze_df.writeStream.format("delta")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .outputMode("append")
)

if trigger_once:
    write_stream = write_stream.trigger(once=True)

write_stream.table(bronze_table)

### Optional monitoring & sampling

In [0]:
active = spark.streams.active
display({"activeQueries": [q.name for q in active] or ["None"]})

display(
    spark.read.table(bronze_table)
    .groupBy("_source_file")
    .count()
    .orderBy(F.desc("count"))
    .limit(20)
)

